# 05. Bolsa de Palabras (BoW) y TF-IDF

Los algoritmos de *machine learning* no trabajan con palabras, sino con números. Para que un modelo pueda analizar textos es necesario convertirlos en representaciones numéricas. A este proceso se le denomina **vectorización** y su resultado es un **vector de características** que resume el contenido de cada documento de forma numérica.

En este notebook estudiaremos las dos técnicas clásicas más utilizadas:

- **Bag of Words (BoW):** representa cada documento contando cuántas veces aparece cada palabra del vocabulario.
- **TF-IDF:** mejora el BoW asignando mayor peso a las palabras características de un documento y menor peso a las palabras comunes en todo el corpus.

Para cada técnica veremos primero la teoría, luego la implementaremos para las dos librerías de las que haremos usos en este tema como son: **scikit-learn** y **Gensim**, explicando sus parámetros más relevantes.

> **Nota:** estas representaciones ignoran deliberadamente el orden de las palabras dentro del documento. En unidades posteriores veremos cómo los *embeddings* neurales superan esta limitación capturando también el significado semántico y las relaciones contextuales.


---

## 0. Corpus de trabajo

Para mostrar los ejemplos vamos a trabajar con un corpus con once documentos sobre fútbol, baloncesto y tenis, usados para demostrar el efecto de los parámetros con un conjunto más realista.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

pd.set_option('display.max_rows', None)     # muestra todas las filas
pd.set_option('display.max_columns', None)  # muestra todas las columnas
pd.set_option('display.width', None)        # no trunca el ancho

# Corpus principal: tres temáticas deportivas (fútbol, baloncesto, tenis)
corpus = [
    # Fútbol
    'gol gol gol messi messi messi messi ronaldo ronaldo liga liga champions champions',
    'futbol futbol futbol gol gol portero portero delantero delantero penalti estadio',
    'gol gol liga liga liga champions champions copa copa messi messi ronaldo ronaldo neymar neymar',
    'penalti penalti arbitro arbitro arbitro arbitro balon balon futbol futbol gol gol estadio estadio',
    # Baloncesto
    'canasta canasta triple triple rebote rebote gol nba nba lebron lebron jordan jordan',
    'baloncesto baloncesto nba nba euroliga euroliga pivot pivot base base alero alero',
    'triple triple canasta canasta lebron jordan jordan gol pabellon pabellon rebote rebote mate mate',
    'nba nba nba lebron lebron canasta canasta triple baloncesto pabellon pabellon euroliga',
    # Tenis
    'tenis tenis nadal nadal djokovic djokovic federer federer gol wimbledon roland_garros saque saque',
    'raqueta raqueta saque saque set set game game nadal pelota pelota gol torneo torneo',
    'wimbledon wimbledon roland_garros roland_garros federer federer djokovic djokovic tenis tenis open open'
]

print(f'Corpus principal: {len(corpus)} documentos (fútbol: 4, baloncesto: 4, tenis: 3)')

Corpus principal: 11 documentos (fútbol: 4, baloncesto: 4, tenis: 3)


---

# 1. Bag of Words (BoW) — Bolsa de Palabras

## 1.1 Concepto

El modelo de **Bolsa de Palabras** representa cada documento como un vector numérico donde cada posición corresponde a un término del vocabulario y su valor indica cuántas veces aparece ese término en el documento.

La idea central es:
1. Construir un **vocabulario** con todos los términos únicos del corpus.
2. Representar cada documento como un vector de dimensión igual al tamaño del vocabulario.

Dado el `corpus_intro`:

```
D1: 'el gato come pescado'
D2: 'el perro come carne'
D3: 'el gato y el perro juegan'
```

El vocabulario resultante sería: `{carne, come, el, gato, juegan, perro, pescado, y}`

Y la representación BoW de cada documento:

| | carne | come | el | gato | juegan | perro | pescado | y |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| D1 | 0 | 1 | 1 | 1 | 0 | 0 | 1 | 0 |
| D2 | 1 | 1 | 1 | 0 | 0 | 1 | 0 | 0 |
| D3 | 0 | 0 | 2 | 1 | 1 | 1 | 0 | 1 |

Fíjate en que en D3 la palabra `el` tiene valor 2 porque aparece dos veces.

**Limitaciones del modelo BoW:**
- **Alta dimensionalidad:** en corpus reales el vocabulario puede tener decenas de miles de términos, generando vectores muy largos con muchos ceros (*sparse*).
- **Pérdida del orden:** las frases *'el gato come al perro'* y *'el perro come al gato'* producen exactamente el mismo vector.

---

## 1.2 BoW con scikit-learn: `CountVectorizer`

scikit-learn proporciona la clase `CountVectorizer` que automatiza todo el proceso: construye el vocabulario, vectoriza los documentos y devuelve una matriz *sparse* eficiente.

La matriz resultante tiene una **fila por documento** y una **columna por término del vocabulario**.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(corpus)

# Mostramos el resultado como DataFrame para facilitar la lectura
pd.DataFrame(bow.toarray(), columns=vectorizer.get_feature_names_out(), index=[f'D{i+1}' for i in range(len(corpus))])


,alero,arbitro,balon,baloncesto,base,canasta,champions,copa,delantero,djokovic,estadio,euroliga,federer,futbol,game,gol,jordan,lebron,liga,mate,messi,nadal,nba,neymar,open,pabellon,pelota,penalti,pivot,portero,raqueta,rebote,roland_garros,ronaldo,saque,set,tenis,torneo,triple,wimbledon
D1,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,3,0,0,2,0,4,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0
D2,0,0,0,0,0,0,0,0,2,0,1,0,0,3,0,2,0,0,0,0,0,0,0,0,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,0
D3,0,0,0,0,0,0,2,2,0,0,0,0,0,0,0,2,0,0,3,0,2,0,0,2,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0
D4,0,4,2,0,0,0,0,0,0,0,2,0,0,2,0,2,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0
D5,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,1,2,2,0,0,0,0,2,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,2,0
D6,2,0,0,2,2,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0
D7,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,1,2,1,0,2,0,0,0,0,0,2,0,0,0,0,0,2,0,0,0,0,0,0,2,0
D8,0,0,0,1,0,2,0,0,0,0,0,1,0,0,0,0,0,2,0,0,0,0,3,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0
D9,0,0,0,0,0,0,0,0,0,2,0,0,2,0,0,1,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,1,0,2,0,2,0,0,1
D10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,1,0,0,0,0,0,1,0,0,0,0,2,0,0,0,2,0,0,0,2,2,0,2,0,0


---

## 1.3 Parámetros principales de `CountVectorizer`

### `max_features` — tamaño máximo del vocabulario

Limita el vocabulario a las `N` palabras más frecuentes del corpus. Es muy útil para reducir la dimensionalidad cuando el vocabulario es muy grande.

- `max_features=None` (valor por defecto): se incluyen todos los términos.
- `max_features=6`: solo se conservan los 6 términos más frecuentes.

### `min_df` — frecuencia mínima de documento

Excluye del vocabulario los términos que aparecen en **menos documentos** que el umbral indicado. Sirve para eliminar palabras muy raras o con errores tipográficos que no aportan información generalizable.

- Si es un **entero**: mínimo número de documentos en los que debe aparecer el término (`min_df=2` → el término debe estar en al menos 2 documentos).
- Si es un **float entre 0 y 1**: proporción mínima de documentos (`min_df=0.2` → el término debe aparecer en al menos el 20 % de los documentos).

### `max_df` — frecuencia máxima de documento

Excluye del vocabulario los términos que aparecen en **más documentos** que el umbral indicado. Funciona como un filtro de *stopwords* automático: palabras tan comunes que aparecen en casi todos los documentos tienen poco poder discriminativo.

- Si es un **entero**: máximo número de documentos en los que puede aparecer el término.
- Si es un **float entre 0 y 1**: proporción máxima de documentos (`max_df=0.8` → el término no puede aparecer en más del 80 % de los documentos).

### `stop_words` — eliminación de palabras vacías

Aunque esta tarea debe hacerse en el proceso de normalización, `CountVectorizer()` permite excluir automáticamente del vocabulario una lista de *stopwords* (palabras muy frecuentes y poco informativas). Solo dispone de stop words en inglés pero se le puede pasar una lista de stops words para que las elimine.

- `stop_words='english'`: usa la lista interna de scikit-learn para inglés.
- `stop_words=[lista]`: acepta una lista personalizada para cualquier idioma.


Veamos a continuación algunos ejemplos del número de palabras que tendríamos en la bolsa de palabras en función de los parámetros vistos.

In [3]:

# ── max_features ──────────────────────────────────────────────────────────────
vec_todos = CountVectorizer()
vec_todos.fit(corpus)
print(f'Vocabulario completo ({len(vec_todos.get_feature_names_out())} términos):')
print(sorted(vec_todos.get_feature_names_out()))
print()

vec_top6 = CountVectorizer(max_features=6)
vec_top6.fit(corpus)
print(f'max_features=6 → {len(vec_top6.get_feature_names_out())} términos:')
print(sorted(vec_top6.get_feature_names_out()))
print()

# ── min_df y max_df ───────────────────────────────────────────────────────────
vec_mindf = CountVectorizer(min_df=2)
vec_mindf.fit(corpus)
print(f'min_df=2 → {len(vec_mindf.get_feature_names_out())} términos:')
print(sorted(vec_mindf.get_feature_names_out()))
print()

vec_maxdf = CountVectorizer(max_df=0.2)
vec_maxdf.fit(corpus)
print(f'max_df=0.2 → {len(vec_maxdf.get_feature_names_out())} términos:')
print(sorted(vec_maxdf.get_feature_names_out()))
print()

vec_combinado = CountVectorizer(min_df=2, max_df=0.2)
vec_combinado.fit(corpus)
print(f'min_df=2 + max_df=0.6 → {len(vec_combinado.get_feature_names_out())} términos:')
print(sorted(vec_combinado.get_feature_names_out()))
print()


Vocabulario completo (40 términos):
['alero', 'arbitro', 'balon', 'baloncesto', 'base', 'canasta', 'champions', 'copa', 'delantero', 'djokovic', 'estadio', 'euroliga', 'federer', 'futbol', 'game', 'gol', 'jordan', 'lebron', 'liga', 'mate', 'messi', 'nadal', 'nba', 'neymar', 'open', 'pabellon', 'pelota', 'penalti', 'pivot', 'portero', 'raqueta', 'rebote', 'roland_garros', 'ronaldo', 'saque', 'set', 'tenis', 'torneo', 'triple', 'wimbledon']

max_features=6 → 6 términos:
['canasta', 'gol', 'lebron', 'liga', 'messi', 'nba']

min_df=2 → 24 términos:
['baloncesto', 'canasta', 'champions', 'djokovic', 'estadio', 'euroliga', 'federer', 'futbol', 'gol', 'jordan', 'lebron', 'liga', 'messi', 'nadal', 'nba', 'pabellon', 'penalti', 'rebote', 'roland_garros', 'ronaldo', 'saque', 'tenis', 'triple', 'wimbledon']

max_df=0.2 → 35 términos:
['alero', 'arbitro', 'balon', 'baloncesto', 'base', 'champions', 'copa', 'delantero', 'djokovic', 'estadio', 'euroliga', 'federer', 'futbol', 'game', 'jordan', 'liga

### `ngram_range` — n-gramas

Por defecto, `CountVectorizer` trabaja con **unigramas** (palabras individuales). El parámetro `ngram_range` permite incluir también **bigramas** (pares de palabras consecutivas) o **trigramas**.

Un bigrama captura contexto local: por ejemplo, *'no me gusta'* y *'me gusta'* son dos bigramas distintos con significados opuestos que el modelo BoW estándar no distingue.

- `ngram_range=(1, 1)`: solo unigramas (valor por defecto).
- `ngram_range=(1, 2)`: unigramas y bigramas.
- `ngram_range=(2, 2)`: solo bigramas.

In [4]:
# Solo unigramas
vec_uni = CountVectorizer(ngram_range=(1, 1))
vec_uni.fit(corpus)
print(f'Unigramas ({len(vec_uni.get_feature_names_out())} términos):')
print(sorted(vec_uni.get_feature_names_out()))
print()

# Unigramas + bigramas
vec_bigram = CountVectorizer(ngram_range=(1, 2))
vec_bigram.fit(corpus)
print(f'Unigramas + bigramas ({len(vec_bigram.get_feature_names_out())} términos):')
print(sorted(vec_bigram.get_feature_names_out()))
print()

# Solo bigramas
vec_solo_bi = CountVectorizer(ngram_range=(2, 2))
vec_solo_bi.fit(corpus)
print(f'Solo bigramas ({len(vec_solo_bi.get_feature_names_out())} términos):')
print(sorted(vec_solo_bi.get_feature_names_out()))

Unigramas (40 términos):
['alero', 'arbitro', 'balon', 'baloncesto', 'base', 'canasta', 'champions', 'copa', 'delantero', 'djokovic', 'estadio', 'euroliga', 'federer', 'futbol', 'game', 'gol', 'jordan', 'lebron', 'liga', 'mate', 'messi', 'nadal', 'nba', 'neymar', 'open', 'pabellon', 'pelota', 'penalti', 'pivot', 'portero', 'raqueta', 'rebote', 'roland_garros', 'ronaldo', 'saque', 'set', 'tenis', 'torneo', 'triple', 'wimbledon']

Unigramas + bigramas (136 términos):
['alero', 'alero alero', 'arbitro', 'arbitro arbitro', 'arbitro balon', 'balon', 'balon balon', 'balon futbol', 'baloncesto', 'baloncesto baloncesto', 'baloncesto nba', 'baloncesto pabellon', 'base', 'base alero', 'base base', 'canasta', 'canasta canasta', 'canasta lebron', 'canasta triple', 'champions', 'champions champions', 'champions copa', 'copa', 'copa copa', 'copa messi', 'delantero', 'delantero delantero', 'delantero penalti', 'djokovic', 'djokovic djokovic', 'djokovic federer', 'djokovic tenis', 'estadio', 'estadio 

### Resumen de parámetros de `CountVectorizer`

| Parámetro | Tipo | Descripción |
|---|---|---|
| `max_features` | int o None | Limita el vocabulario a los N términos más frecuentes |
| `min_df` | int o float | Excluye términos que aparecen en menos de N documentos |
| `max_df` | int o float | Excluye términos que aparecen en más de N documentos |
| `ngram_range` | tuple (min, max) | Rango de n-gramas a incluir |
| `stop_words` | str o lista | Lista de palabras a excluir del vocabulario |
| `lowercase` | bool | Convierte el texto a minúsculas antes de tokenizar (True por defecto) |

---

# 2. TF-IDF (*Term Frequency - Inverse Document Frequency*)

## 2.1 Concepto y fórmulas

TF-IDF es una medida estadística que evalúa la **importancia relativa** de un término dentro de un documento en el contexto de una colección de documentos.

La intuición es la siguiente: una palabra que aparece frecuentemente en un documento es importante para ese documento; pero si esa misma palabra aparece en casi todos los documentos del corpus, su capacidad para distinguir ese documento del resto es baja. TF-IDF combina ambos aspectos en una única puntuación.


### TF (Term Frequency)

Mide la frecuencia normalizada con la que aparece un término $t$ en un documento $d$. Usando la variante logarítmica, que suaviza el efecto de las repeticiones muy altas:

$$tf(t, d) = 1 + \log\left(\frac{f_{t,d}}{|d|}\right)$$

donde $f_{t,d}$ es el número de veces que aparece el término $t$ en el documento $d$, y $|d|$ es el número total de términos del documento.

### IDF (Inverse Document Frequency)

Penaliza los términos que aparecen en muchos documentos del corpus. Usando la variante suavizada para evitar divisiones por cero:

$$idf(t, D) = \log\left(1 + \frac{N}{n_t}\right)$$

donde $N$ es el número total de documentos en el corpus $D$ y $n_t$ es el número de documentos que contienen el término $t$. Cuantos más documentos contengan el término, menor será su IDF.

### TF-IDF

$$tfidf(t, d, D) = tf(t, d) \cdot idf(t, D)$$

Un valor alto de TF-IDF indica que el término es frecuente en ese documento concreto pero poco frecuente en el resto del corpus: es un término **característico y discriminativo** de ese documento.

## 2.2 TF-IDF con scikit-learn: `TfidfVectorizer`

`TfidfVectorizer` automatiza el cálculo de TF-IDF. Acepta los mismos parámetros que `CountVectorizer` (`max_features`, `min_df`, `max_df`, `ngram_range`, `stop_words`) más algunos adicionales propios.

### `sublinear_tf` — escalado logarítmico del TF

Si `sublinear_tf=True`, sustituye la frecuencia bruta $f_{t,d}$ por $1 + \log(f_{t,d})$ para reducir el peso de las palabras con repeticiones muy altas. Es especialmente útil en documentos largos donde algunas palabras se repiten muchas veces.

### `norm` — normalización del vector

- `norm='l2'` (valor por defecto): normaliza cada vector de documento para que su norma euclídea sea 1. Hace que los documentos sean comparables independientemente de su longitud.
- `norm=None`: devuelve los valores TF-IDF sin normalizar.

### `use_idf` — desactivar el componente IDF

Si `use_idf=False`, el vectorizador solo calcula el TF (equivale a un `CountVectorizer` con normalización). Útil para comparar el efecto del IDF.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
matriz_tfidf = tfidf.fit_transform(corpus)

pd.DataFrame(matriz_tfidf.toarray().round(2), columns=tfidf.get_feature_names_out(), index=[f'D{i+1}' for i in range(len(corpus))])


,alero,arbitro,balon,baloncesto,base,canasta,champions,copa,delantero,djokovic,estadio,euroliga,federer,futbol,game,gol,jordan,lebron,liga,mate,messi,nadal,nba,neymar,open,pabellon,pelota,penalti,pivot,portero,raqueta,rebote,roland_garros,ronaldo,saque,set,tenis,torneo,triple,wimbledon
D1,0.00,0.00,0.00,0.00,0.00,0.00,0.36,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.29,0.00,0.00,0.36,0.00,0.72,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.36,0.00,0.00,0.00,0.00,0.00,0.00
D2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.49,0.00,0.21,0.00,0.00,0.62,0.00,0.22,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.21,0.00,0.49,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
D3,0.00,0.00,0.00,0.00,0.00,0.00,0.35,0.41,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.19,0.00,0.00,0.52,0.00,0.35,0.00,0.00,0.41,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.35,0.00,0.00,0.00,0.00,0.00,0.00
D4,0.00,0.73,0.37,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.31,0.00,0.00,0.31,0.00,0.17,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
D5,0.00,0.00,0.00,0.00,0.00,0.39,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.12,0.44,0.39,0.00,0.00,0.00,0.00,0.39,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.44,0.00,0.00,0.00,0.00,0.00,0.00,0.39,0.00
D6,0.45,0.00,0.00,0.38,0.45,0.00,0.00,0.00,0.00,0.00,0.00,0.38,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.34,0.00,0.00,0.00,0.00,0.00,0.45,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
D7,0.00,0.00,0.00,0.00,0.00,0.35,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.11,0.40,0.18,0.00,0.47,0.00,0.00,0.00,0.00,0.00,0.40,0.00,0.00,0.00,0.00,0.00,0.40,0.00,0.00,0.00,0.00,0.00,0.00,0.35,0.00
D8,0.00,0.00,0.00,0.22,0.00,0.39,0.00,0.00,0.00,0.00,0.00,0.22,0.00,0.00,0.00,0.00,0.00,0.39,0.00,0.00,0.00,0.00,0.59,0.00,0.00,0.45,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.20,0.00
D9,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.42,0.00,0.00,0.42,0.00,0.00,0.11,0.00,0.00,0.00,0.00,0.00,0.42,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.21,0.00,0.42,0.00,0.42,0.00,0.00,0.21
D10,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.41,0.09,0.00,0.00,0.00,0.00,0.00,0.17,0.00,0.00,0.00,0.00,0.41,0.00,0.00,0.00,0.41,0.00,0.00,0.00,0.35,0.41,0.00,0.41,0.00,0.00


---

# 3. Comparación BoW vs TF-IDF sobre el mismo corpus

Construimos ambas representaciones sobre el corpus principal usando exactamente los mismos parámetros de filtrado para ver cómo difieren los valores.

In [6]:
params_comunes = dict(max_features=8, min_df=2)

bow_comp   = CountVectorizer(**params_comunes)
tfidf_comp = TfidfVectorizer(**params_comunes)

m_bow   = bow_comp.fit_transform(corpus)
m_tfidf = tfidf_comp.fit_transform(corpus)

# Usamos el mismo vocabulario para comparar columna a columna
palabras_bow = bow_comp.get_feature_names_out()

print('--- BoW (frecuencias) ---')
df_cmp_bow = pd.DataFrame(
    m_bow.toarray(),
    columns=palabras_bow,
    index=[f'Doc {i+1}' for i in range(len(corpus))]
)
print(df_cmp_bow.to_string())

print()
print('--- TF-IDF (pesos normalizados) ---')
df_cmp_tfidf = pd.DataFrame(
    m_tfidf.toarray().round(3),
    columns=tfidf_comp.get_feature_names_out(),
    index=[f'Doc {i+1}' for i in range(len(corpus))]
)
print(df_cmp_tfidf.to_string())

--- BoW (frecuencias) ---
        canasta  futbol  gol  lebron  liga  messi  nba  triple
Doc 1         0       0    3       0     2      4    0       0
Doc 2         0       3    2       0     0      0    0       0
Doc 3         0       0    2       0     3      2    0       0
Doc 4         0       2    2       0     0      0    0       0
Doc 5         2       0    1       2     0      0    2       2
Doc 6         0       0    0       0     0      0    2       0
Doc 7         2       0    1       1     0      0    0       2
Doc 8         2       0    0       2     0      0    3       1
Doc 9         0       0    1       0     0      0    0       0
Doc 10        0       0    1       0     0      0    0       0
Doc 11        0       0    0       0     0      0    0       0

--- TF-IDF (pesos normalizados) ---
        canasta  futbol    gol  lebron   liga  messi    nba  triple
Doc 1     0.000   0.000  0.340   0.000  0.421  0.841  0.000   0.000
Doc 2     0.000   0.941  0.339   0.000  0.000

---

# 4. Gensim — ejemplo de referencia

Gensim es otra librería popular para NLP que implementa BoW y TF-IDF con una API diferente a scikit-learn. Los conceptos son idénticos; lo que cambia es la forma de llamar a las funciones.

La diferencia principal en el formato de salida es:
- **scikit-learn** devuelve una **matriz densa o sparse** donde cada fila es un documento.
- **Gensim** devuelve una **lista de listas de tuplas** `(id_término, valor)`, almacenando únicamente los términos con valor distinto de cero (formato sparse explícito).

## 4.1 BoW con Gensim

Gensim trabaja con un objeto `Dictionary` que asigna un identificador numérico a cada término del vocabulario. Después, `doc2bow()` convierte cada documento en una lista de tuplas `(id, frecuencia)`.

In [7]:
import gensim
from gensim import corpora

# Tokenizamos con split() simple (en proyectos reales se usa spaCy o NLTK)
tokens = [doc.split() for doc in corpus]

# Construimos el diccionario de términos
diccionario = corpora.Dictionary(tokens)

print('Diccionario (término → id):')
print(diccionario.token2id)
print()

# Construimos el BoW de cada documento: lista de tuplas (id_termino, frecuencia)
bow_gensim = [diccionario.doc2bow(token) for token in tokens]

print('BoW en formato Gensim (id_termino, frecuencia):')
for i, doc_bow in enumerate(bow_gensim):
    terminos = [(diccionario[id_t], freq) for id_t, freq in doc_bow]
    print(f'  D{i+1}: {terminos}')

Diccionario (término → id):
{'champions': 0, 'gol': 1, 'liga': 2, 'messi': 3, 'ronaldo': 4, 'delantero': 5, 'estadio': 6, 'futbol': 7, 'penalti': 8, 'portero': 9, 'copa': 10, 'neymar': 11, 'arbitro': 12, 'balon': 13, 'canasta': 14, 'jordan': 15, 'lebron': 16, 'nba': 17, 'rebote': 18, 'triple': 19, 'alero': 20, 'baloncesto': 21, 'base': 22, 'euroliga': 23, 'pivot': 24, 'mate': 25, 'pabellon': 26, 'djokovic': 27, 'federer': 28, 'nadal': 29, 'roland_garros': 30, 'saque': 31, 'tenis': 32, 'wimbledon': 33, 'game': 34, 'pelota': 35, 'raqueta': 36, 'set': 37, 'torneo': 38, 'open': 39}

BoW en formato Gensim (id_termino, frecuencia):
  D1: [('champions', 2), ('gol', 3), ('liga', 2), ('messi', 4), ('ronaldo', 2)]
  D2: [('gol', 2), ('delantero', 2), ('estadio', 1), ('futbol', 3), ('penalti', 1), ('portero', 2)]
  D3: [('champions', 2), ('gol', 2), ('liga', 3), ('messi', 2), ('ronaldo', 2), ('copa', 2), ('neymar', 2)]
  D4: [('gol', 2), ('estadio', 2), ('futbol', 2), ('penalti', 2), ('arbitro', 

## 4.2 TF-IDF con Gensim

El modelo TF-IDF de Gensim toma como entrada el corpus en formato BoW y devuelve los pesos TF-IDF en el mismo formato de tuplas `(id_término, peso_tfidf)`.

In [8]:
from gensim.models import TfidfModel

# Construimos el modelo TF-IDF a partir del corpus BoW
modelo_tfidf = TfidfModel(bow_gensim, dictionary=diccionario, normalize=True)

# Transformamos el corpus
tfidf_gensim = [modelo_tfidf[doc_bow] for doc_bow in bow_gensim]

print('TF-IDF en formato Gensim (id_termino, peso):')
for i, doc_tfidf in enumerate(tfidf_gensim):
    terminos = [(diccionario[id_t], round(peso, 4)) for id_t, peso in doc_tfidf]
    print(f'  D{i+1}: {terminos}')

TF-IDF en formato Gensim (id_termino, peso):
  D1: [('champions', np.float64(0.3759)), ('gol', np.float64(0.1053)), ('liga', np.float64(0.3759)), ('messi', np.float64(0.7517)), ('ronaldo', np.float64(0.3759))]
  D2: [('gol', np.float64(0.0719)), ('delantero', np.float64(0.5417)), ('estadio', np.float64(0.1926)), ('futbol', np.float64(0.5777)), ('penalti', np.float64(0.1926)), ('portero', np.float64(0.5417))]
  D3: [('champions', np.float64(0.3289)), ('gol', np.float64(0.0614)), ('liga', np.float64(0.4934)), ('messi', np.float64(0.3289)), ('ronaldo', np.float64(0.3289)), ('copa', np.float64(0.4627)), ('neymar', np.float64(0.4627))]
  D4: [('gol', np.float64(0.052)), ('estadio', np.float64(0.2781)), ('futbol', np.float64(0.2781)), ('penalti', np.float64(0.2781)), ('arbitro', np.float64(0.7824)), ('balon', np.float64(0.3912))]
  D5: [('gol', np.float64(0.0449)), ('canasta', np.float64(0.3662)), ('jordan', np.float64(0.4804)), ('lebron', np.float64(0.3662)), ('nba', np.float64(0.3662)), ('

---

## 5. Conclusiones

En este notebook hemos visto las dos técnicas clásicas de vectorización de texto:

- El modelo **BoW** (`CountVectorizer`) representa cada documento como un vector de frecuencias. Es sencillo y eficiente, pero no distingue entre palabras comunes y palabras discriminativas.

- **TF-IDF** (`TfidfVectorizer`) mejora el BoW penalizando las palabras que aparecen en muchos documentos del corpus. El resultado es una representación que da más peso a los términos característicos de cada documento.

- Ambas clases de scikit-learn comparten los mismos parámetros de filtrado: `max_features` para limitar el vocabulario, `min_df` y `max_df` para controlar la frecuencia de los términos, `ngram_range` para capturar contexto local mediante bigramas o trigramas, y `stop_words` para eliminar palabras vacías.

- `TfidfVectorizer` añade `sublinear_tf` para suavizar el efecto de las repeticiones muy altas, y `norm` para normalizar los vectores y hacer los documentos comparables con independencia de su longitud.

- **Gensim** implementa los mismos conceptos con una API diferente: trabaja con diccionarios y devuelve los vectores en formato de tuplas `(id, valor)`, lo que es eficiente para corpus muy grandes.

Estas representaciones vectoriales son el punto de partida para tareas de clasificación de texto, análisis de sentimientos o recuperación de información con algoritmos de aprendizaje automático clásicos.